# 00 — Kiểm tra dữ liệu và hợp đồng định danh

Notebook này đọc cấu hình, ánh xạ lớp, số lượng mẫu và dấu vết nguồn của hai luồng đang được hỗ trợ: detector 9 thương hiệu và embedding 5 SKU AXE. Các bước quét ảnh đầy đủ, preflight và đánh giá test đều là opt-in; mở notebook hoặc chạy các ô hiển thị không khởi động huấn luyện.

Phạm vi mặc định chỉ là `train` và `val`. Tên nhóm theo file không chứng minh các ảnh độc lập theo phiên chụp; đó vẫn là một giới hạn cần đánh giá thủ công.

## Khởi chạy

Chạy từ thư mục gốc của repository trong PowerShell:

```powershell
Set-Location D:/code/anhoa/train_embedding
./.venv/Scripts/Activate.ps1
jupyter lab notebooks/00_dataset_audit.ipynb
```

Notebook không có ô cài package. Nếu kernel thiếu Jupyter, matplotlib hoặc IPython, hãy chuẩn bị môi trường bên ngoài notebook rồi khởi động lại kernel.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() or not (PROJECT_ROOT / "configs").is_dir():
    raise RuntimeError("Hãy khởi chạy Jupyter từ thư mục gốc của repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display

from src.catalog.profiles import load_product_profile
from src.config import load_brand_registry, resolve_repo_path
from src.data import load_yolo_class_names
from src.data import load_dataset_sources
from src.training.embedding_data import audit_training_readiness, path_key, read_csv
from src.training.yolo import build_generic_preflight, read_yolo_config
from src.utils.config import load_config

def show_table(rows: list[dict], title: str) -> None:
    display(Markdown(f"**{title}**"))
    if not rows:
        display(Markdown("_Không có dòng dữ liệu._"))
        return
    columns = list(rows[0])
    clean = lambda value: str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    lines.extend("| " + " | ".join(clean(row.get(column, "")) for column in columns) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

## Chọn cấu hình và cờ thực thi

Các đường dẫn đều được giải theo repository root qua `src.config`. `val` là split đánh giá mặc định. Chỉ đổi `RUN_TEST_EVALUATION` sau khi đã chốt checkpoint và có chủ đích mở holdout test.

In [ ]:
BRAND_REGISTRY_PATH = "configs/brands.yaml"
PRODUCT_PROFILE_NAME = "axe5"
DETECTOR_CONFIG_PATH = "configs/brand_9brands_yolo.yaml"
EMBEDDING_CONFIG_PATH = "configs/axe_5sku_100ep_classification.yaml"
DEFAULT_EVALUATION_SPLIT = "val"

RUN_DETECTOR_PREFLIGHT = False
RUN_EMBEDDING_READINESS = False
RUN_TEST_EVALUATION = False

assert DEFAULT_EVALUATION_SPLIT == "val"
show_table(
    [
        {"thành phần": "brand registry", "lựa chọn": BRAND_REGISTRY_PATH},
        {"thành phần": "detector", "lựa chọn": DETECTOR_CONFIG_PATH},
        {"thành phần": "embedding", "lựa chọn": EMBEDDING_CONFIG_PATH},
        {"thành phần": "product profile", "lựa chọn": PRODUCT_PROFILE_NAME},
        {"thành phần": "split mặc định", "lựa chọn": DEFAULT_EVALUATION_SPLIT},
    ],
    "Lựa chọn hiện tại",
)

## Registry, thứ tự lớp và item code

Các bảng dưới đây đọc trực tiếp registry thương hiệu, dataset source registry, `data.yaml` của detector và product profile AXE. Item code được giữ ở dạng chuỗi. Guard Blue Lavender phải tiếp tục phân biệt spray `64823658` với body wash `64863894`.

In [ ]:
registry = load_brand_registry(resolve_repo_path(BRAND_REGISTRY_PATH))
source_registry = load_dataset_sources()
profile = load_product_profile(PRODUCT_PROFILE_NAME)
detector_config = read_yolo_config(resolve_repo_path(DETECTOR_CONFIG_PATH))
detector_data_yaml = resolve_repo_path(detector_config["data"])
dataset_class_names = load_yolo_class_names(detector_data_yaml)
registry_class_names = registry.names("brand_9brands")

show_table(
    [
        {"class_id": brand.id, "brand": brand.name, "aliases": ", ".join(brand.aliases) or "—"}
        for brand in registry.profile("brand_9brands")
    ],
    "Registry detector 9 thương hiệu",
)
show_table(
    [
        {
            "source": source.name,
            "format": source.annotation_format.value,
            "class_mapping_source": source.class_mapping_source,
            "sha256": source.sha256 or "chưa cấu hình",
        }
        for source in source_registry.sources
    ],
    "Dataset source registry",
)
show_table(
    [
        {
            "class_index": row.class_index,
            "legacy_class_name": row.legacy_class_name,
            "item_code": row.item_code,
            "kiểu": type(row.item_code).__name__,
        }
        for row in profile.classes
    ],
    "Hợp đồng checkpoint → item code cho AXE 5 SKU",
)
show_table(
    [
        {"nguồn": name, "class_order": " → ".join(values)}
        for name, values in (("registry", registry_class_names), ("data.yaml", dataset_class_names))
    ],
    "Đối chiếu thứ tự lớp detector",
)
assert tuple(dataset_class_names) == tuple(registry_class_names)
assert profile.identity_guard("blue_lavender")["spray"] != profile.identity_guard("blue_lavender")["body_wash"]

## Số lượng và dấu vết detector

Ô này chỉ đọc báo cáo đã sinh; không mở ảnh và không đánh giá checkpoint. Biểu đồ chỉ dùng `train` và `val`. Metadata test có thể tồn tại trong file lịch sử nhưng không được dùng để tính metric ở đây.

In [ ]:
checks_path = detector_data_yaml.parent / "dataset_checks.json"
checks = json.loads(checks_path.read_text(encoding="utf-8"))
split_rows = [
    {
        "split": split,
        "images": checks["splits"][split]["images"],
        "labels": checks["splits"][split]["labels"],
        "instances": checks["splits"][split]["instances"],
    }
    for split in ("train", "val")
]
show_table(split_rows, "Quy mô detector (không mở test)")

brand_count_rows = [
    {
        "brand": brand,
        "train_images": checks["splits"]["train"]["per_brand"][brand]["images"],
        "train_instances": checks["splits"]["train"]["per_brand"][brand]["instances"],
        "val_images": checks["splits"]["val"]["per_brand"][brand]["images"],
        "val_instances": checks["splits"]["val"]["per_brand"][brand]["instances"],
    }
    for brand in registry_class_names
]
show_table(brand_count_rows, "Phân bố theo thương hiệu")

show_table(
    [
        {"kiểm tra": "status", "giá trị": checks.get("status")},
        {"kiểm tra": "cross_split_family_overlap", "giá trị": checks.get("cross_split_family_overlap")},
        {"kiểm tra": "exact_image_cross_split_overlap", "giá trị": checks.get("exact_image_cross_split_overlap")},
        {"kiểm tra": "quarantined_hashes_in_splits", "giá trị": checks.get("quarantined_hashes_in_splits")},
        {"kiểm tra": "unresolved_annotation_hashes", "giá trị": checks.get("unresolved_annotation_hashes")},
        {"kiểm tra": "test_split_evaluated", "giá trị": checks.get("test_split_evaluated")},
    ],
    "Các bất biến đã ghi nhận",
)
provenance_paths = [
    checks_path,
    detector_data_yaml.parent / "dataset_report.json",
    detector_data_yaml.parent / "source_family_manifest.json",
    detector_data_yaml.parent / "provenance.csv",
]
show_table(
    [{"artifact": path.relative_to(PROJECT_ROOT).as_posix(), "tồn tại": path.is_file(), "bytes": path.stat().st_size if path.is_file() else 0} for path in provenance_paths],
    "Artifact provenance",
)

In [ ]:
import matplotlib.pyplot as plt

brands = [row["brand"] for row in brand_count_rows]
positions = list(range(len(brands)))
width = 0.42
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar([x - width / 2 for x in positions], [row["train_instances"] for row in brand_count_rows], width, label="train")
ax.bar([x + width / 2 for x in positions], [row["val_instances"] for row in brand_count_rows], width, label="val")
ax.set_xticks(positions, brands, rotation=35, ha="right")
ax.set_ylabel("Số bbox")
ax.set_title("Phân bố bbox detector theo thương hiệu")
ax.legend()
fig.tight_layout()
plt.show()

## Số lượng và provenance của AXE embedding

Manifest được đọc bằng helper đang dùng trong luồng huấn luyện. Chỉ `train.csv` và `val.csv` được mở; `test.csv` vẫn đóng.

In [ ]:
embedding_config = load_config(EMBEDDING_CONFIG_PATH)
crops_root = resolve_repo_path(embedding_config["dataset"]["path"])
clean_root = resolve_repo_path(embedding_config["dataset"]["source_coco_path"])
manifest_root = resolve_repo_path(embedding_config["dataset"]["split_dir"])
manifest_rows = {
    "train": read_csv(manifest_root / "train.csv"),
    "val": read_csv(manifest_root / "val.csv"),
}
crop_provenance = read_csv(crops_root / "crop_manifest.csv")
provenance_by_crop = {path_key(row["generated_crop_path"]): row for row in crop_provenance}
axe_rows = []
for split, rows in manifest_rows.items():
    counts = Counter(row["class_name"] for row in rows)
    for identity in profile.classes:
        selected = [provenance_by_crop[path_key(row["image_path"])] for row in rows if row["class_name"] == identity.legacy_class_name]
        axe_rows.append(
            {
                "split": split,
                "class_index": identity.class_index,
                "item_code": identity.item_code,
                "crops": counts[identity.legacy_class_name],
                "source_families": len({row["family_id"] for row in selected}),
                "source_zips": len({row["source_zip"] for row in selected}),
            }
        )
show_table(axe_rows, "AXE train/val theo item code")
show_table(
    [
        {"artifact": path.relative_to(PROJECT_ROOT).as_posix(), "tồn tại": path.is_file(), "bytes": path.stat().st_size if path.is_file() else 0}
        for path in (crops_root / "crop_manifest.csv", clean_root / "source_family_manifest.json", clean_root / "build_report.json")
    ],
    "Nguồn provenance AXE",
)

## Preflight/readiness có chủ đích

Detector preflight kiểm tra cấu hình, class order, policy quarantine và không cho ghi đè run. AXE readiness xác minh ảnh, hash và family overlap nhưng chỉ đối với train/validation. Hai bước có thể mất thời gian nên mặc định tắt.

In [ ]:
if RUN_DETECTOR_PREFLIGHT:
    detector_plan, detector_summary = build_generic_preflight(DETECTOR_CONFIG_PATH)
    show_table([{"trường": key, "giá trị": value} for key, value in detector_summary.items()], "Detector preflight")
else:
    print("Detector preflight đang tắt: RUN_DETECTOR_PREFLIGHT=False")

if RUN_EMBEDDING_READINESS:
    embedding_readiness = audit_training_readiness(crops_root, clean_root)
    show_table(
        [
            {"split": split, "crops": count, "source_families": embedding_readiness["source_family_counts"][split]}
            for split, count in embedding_readiness["crop_counts"].items()
        ],
        "AXE readiness (train/validation)",
    )
    assert embedding_readiness["test_queries_loaded"] is False
else:
    print("AXE readiness đang tắt: RUN_EMBEDDING_READINESS=False")

## Test holdout — opt-in riêng

Ô kế tiếp gọi đúng CLI audit lịch sử; CLI này quét toàn bộ dữ liệu, nạp checkpoint và **đánh giá test**. Chỉ đặt `RUN_TEST_EVALUATION=True` sau khi đã hoàn tất lựa chọn mô hình bằng validation. Output mới được yêu cầu để tránh ghi đè báo cáo cũ.

In [ ]:
test_audit_output = resolve_repo_path("outputs/notebook_dataset_audit_test")
test_audit_command = [
    sys.executable,
    str(resolve_repo_path("scripts/audit_dataset_leakage.py")),
    "--dataset-root", str(clean_root),
    "--crops-root", str(crops_root),
    "--checkpoint", str(resolve_repo_path("checkpoints/axe_5sku/classification_100ep_s42/best_model.pth")),
    "--config", str(resolve_repo_path(EMBEDDING_CONFIG_PATH)),
    "--output-dir", str(test_audit_output),
    "--device", "auto",
]
print("Lệnh test đã chuẩn bị:", subprocess.list2cmdline(test_audit_command))
if RUN_TEST_EVALUATION:
    if test_audit_output.exists():
        raise FileExistsError(f"Chọn output mới; không ghi đè: {test_audit_output}")
    subprocess.run(test_audit_command, cwd=PROJECT_ROOT, check=True)
else:
    print("Không đánh giá test: RUN_TEST_EVALUATION=False")

## Diễn giải kết quả

- Class order phải khớp tuyệt đối giữa registry, dataset và checkpoint; không sắp xếp lại ID sau khi đã huấn luyện.
- Quarantine không được trộn lại vào split đang hoạt động; xung đột annotation cần review thủ công.
- Số crop cao nhưng ít source family là dấu hiệu tương quan mạnh, không phải bằng chứng về đa dạng dữ liệu.
- Chuyển sang `01_detector_training.ipynb` hoặc `02_embedding_training.ipynb` chỉ sau khi preflight/readiness đạt yêu cầu.